## Neo4j Cypher queries

### Neo4j Native Upsert. Matching using id

In [ ]:
def create_podcast_and_episode(tx, podcast_title, episode_data):
    tx.run("""
    // Match an existing podcast by id or title
    MERGE (pod:Podcast)
      ON CREATE SET pod.id = $podcast_title,
                    pod.title = $podcast_title
      ON MATCH SET pod.title = coalesce(pod.title, $podcast_title)
    WITH pod

    // Create or match episode
    MERGE (ep:Episode {number: $ep_number})
    SET ep.name = $ep_name,
        ep.published_date = date($ep_date),
        ep.link = $ep_link,
        ep.description = $ep_description

    // Connect podcast to episode
    MERGE (pod)-[:HAS_EPISODE]->(ep)
    """, 
    podcast_title=podcast_title,
    ep_number=episode_data["number"],
    ep_name=episode_data["name"],
    ep_date=episode_data["published_date"],
    ep_link=episode_data.get("link", ""),
    ep_description=episode_data.get("description", "")
    )

### Using APOC Plugin

In [ ]:
def create_podcast_and_episode(tx, podcast_title, episode_data):
    tx.run("""
    // Try to match an existing podcast by id or title
    OPTIONAL MATCH (p1:Podcast {id: $podcast_title})
    OPTIONAL MATCH (p2:Podcast {title: $podcast_title})
    WITH coalesce(p1, p2) AS pod

    // If not found, create a new podcast
    CALL apoc.do.when(
        pod IS NULL,
        '
        CREATE (newPod:Podcast {id: $podcast_title, title: $podcast_title})
        RETURN newPod
        ',
        '
        RETURN pod AS newPod
        ',
        {pod: pod, podcast_title: $podcast_title}
    ) YIELD value
    WITH value.newPod AS pod

    // Merge or update episode details
    MERGE (ep:Episode {number: $ep_number})
    SET ep.name = $ep_name,
        ep.published_date = date($ep_date),
        ep.link = $ep_link,
        ep.description = $ep_description

    // Connect podcast to episode
    MERGE (pod)-[:HAS_EPISODE]->(ep)
    """,
    podcast_title=podcast_title,
    ep_number=episode_data["number"],
    ep_name=episode_data["name"],
    ep_date=episode_data["published_date"],
    ep_link=episode_data.get("link", ""),
    ep_description=episode_data.get("description", "")
    )

### Create Podcast Uniquness constraint

In [ ]:
CREATE CONSTRAINT podcast_unique_id IF NOT EXISTS
FOR (p:Podcast)
REQUIRE p.id IS UNIQUE;

### Add Topic to Episode connection, COVERED_BY_EPISODE

In [ ]:
MATCH (e:Episode)-[:HAS_TOPIC]->(t:Topic)
// Match every existing relationship from Episode to Topic

MERGE (t)-[:COVERED_BY_EPISODE]->(e)
// MERGE creates the new relationship (Topic -> Episode) 
// if it doesn't already exist.

RETURN count(t) AS TopicsProcessed

### Create Vectror index for Chunk nodes' embedding

In [ ]:
CREATE VECTOR INDEX chunkIndex
FOR (c:Chunk)
ON (c.embedding)
OPTIONS {
  indexConfig: {
    `vector.dimensions`: 1536,
    `vector.similarity_function`: 'cosine'
  }
};

### List all the indexes with their properties

In [ ]:
SHOW INDEXES

### Show indexes of type Vector

In [ ]:
SHOW INDEXES
YIELD name, type, labelsOrTypes, properties, options
WHERE type = 'VECTOR'
RETURN name, labelsOrTypes, properties, options;

### Show specific index by Index Name

In [ ]:
SHOW INDEXES
YIELD name, state, type, entityType, labelsOrTypes, properties, options
WHERE name = 'chunkIndex'
RETURN name, state, type, entityType, labelsOrTypes, properties, options;

####  Check Neo4j Database Version

In [ ]:
// "Neo4j Kernel": ["5.27-aura"] | Edition: "enterprise"
//Cypher: ["5", "25"]
CALL dbms.components() YIELD name, versions, edition
RETURN name, versions, edition;


#### Check GDS Version

In [ ]:
//"2.22.0"
RETURN gds.version() 


#### Existing technologies coverred in an Episode

In [ ]:
MATCH (p:Podcast)-[:HAS_EPISODE]->(e:Episode {number:1836})-[:HAS_TOPIC]->(t:Topic)
OPTIONAL MATCH (t)-[:COVERS_TECHNOLOGY]->(existingTech:Technology)
WITH e, t, collect(existingTech.name) AS existingTechnologies
RETURN 
  e.name AS episode,
  t.name AS topic,
  existingTechnologies


#### Add a new tech to existing Episode

In [ ]:
MATCH (p:Podcast)-[:HAS_EPISODE]->(e:Episode {number:1836})-[:HAS_TOPIC]->(t:Topic)
OPTIONAL MATCH (t)-[:COVERS_TECHNOLOGY]->(existingTech:Technology)
WITH e, t, collect(existingTech.name) AS existingTechnologies
MERGE (newTech:Technology {name: "Agent"})
MERGE (t)-[:COVERS_TECHNOLOGY]->(newTech)
RETURN 
  e.name AS episode,
  t.name AS topic,
  existingTechnologies,
  newTech.name AS addedTechnology;


#### Native projection (does not include any properties) 

In [ ]:
// Project episodes and their content chunks - Not of much use
CALL gds.graph.project('episode-content', 
    ['Episode', 'Chunk'], 
    ['HAS_CHUNK'] 
);

## Cypher Projection, Native Projection: https://neo4j.com/docs/graph-data-science/current/management-ops/graph-list/

## Content based similarity using GDS

#### Content-based similarity, you need a three-step process:

Correct Projection: Project the graph, ensuring the embedding property is included on the Chunk nodes.

Generate Episode Embeddings (FastRP): Use FastRP to aggregate the embeddings of all connected Chunk nodes, creating a single, representative embedding for the Episode node.

Calculate Similarity (Node Similarity): Run Node Similarity on the Episode nodes, comparing the newly generated episode embeddings.

Assumption: I will assume the embedding property on the Chunk node is named embedding. Please adjust this if your property has a different name.

Here are the three sequential GDS queries:

#### Inspect if any of the nodes needed for projection is missing the field being projected (embedding)

In [ ]:
MATCH (n)-[]-()
WHERE 
    (n:Episode OR n:Topic OR n:Concept OR n:Technology OR n:ReferenceLink OR n:Chunk)
    AND n.embedding IS NULL
RETURN 
    n

In [ ]:
MATCH (n)
WHERE 
    (n:Episode OR n:Topic OR n:Concept OR n:Technology OR n:ReferenceLink OR n:Chunk)
    AND n.embedding IS NULL
RETURN 
    labels(n) AS NodeLabel, 
    count(n) AS Count,
    collect(n.id) AS SampleIDs // Assuming nodes have an 'id' property
ORDER BY NodeLabel

#####  Rename the conflicting node label from Concept to Concepts

In [ ]:

MATCH (n:Concept)
WHERE n.id = "Ontologies"
SET n:Concepts       
REMOVE n:Concept
RETURN n

#### List basic information about all graphs in the catalog

In [ ]:
CALL gds.graph.list()
YIELD graphName, nodeCount, relationshipCount
RETURN graphName, nodeCount, relationshipCount
ORDER BY graphName ASC

#### Step 1: Project Episodes with embedding

In [ ]:
CALL gds.graph.project(
'EpisodeSimilarityGraph',
// Node specification: Use map syntax to define node labels and their properties
{
Episode: {
label: 'Episode',
properties: {
embedding: { type: 'LIST OF FLOAT' }
}
},
Topic: {
label: 'Topic',
properties: {
embedding: { type: 'LIST OF FLOAT' }
}
},
Concept: {
label: 'Concept',
properties: {
embedding: { type: 'LIST OF FLOAT' }
}
},
Technology: {
label: 'Technology',
properties: {
embedding: { type: 'LIST OF FLOAT' }
}
},
ReferenceLink: {
label: 'ReferenceLink',
properties: {
embedding: { type: 'LIST OF FLOAT' }
}
},
Chunk: {
label: 'Chunk',
properties: {
embedding: { type: 'LIST OF FLOAT' }
}
}

},
// Relationship specification (as before)
{
HAS_CHUNK: { orientation: 'UNDIRECTED' },
BELONGS_TO_EPISODE: { orientation: 'UNDIRECTED' },
HAS_TOPIC: { orientation: 'UNDIRECTED' },
COVERS_CONCEPT: { orientation: 'UNDIRECTED' },
COVERS_TECHNOLOGY: { orientation: 'UNDIRECTED' },
HAS_REFERENCE_LINK: { orientation: 'UNDIRECTED' }
}
)
YIELD graphName, nodeCount, relationshipCount

#### List extended information about a specific native named graph in the catalog:

In [ ]:
CALL gds.graph.list("EpisodeSimilarityGraph")
YIELD graphName, schemaWithOrientation, configuration
RETURN graphName, schemaWithOrientation, configuration.nodeProjection AS nodeProjection

#### Mutate call to create episodeFastRPEmbedding property in-memory with embeddingDimension (1536 - same as openAI embed_text model, 'text-embedding-3-small')

In [ ]:
CALL gds.fastRP.mutate(
'EpisodeSimilarityGraph', // The name of your projected graph
{
// Mandatory parameter for mutate mode
mutateProperty: 'episodeFastRPEmbedding', // Property name for the resulting vector in-memory

// Feature and Determinism Configuration (Same as write)
embeddingDimension: 1536,
propertyRatio: 1.0,
featureProperties: ["embedding"],
randomSeed: 42,

// Iteration Configuration (Same as write)
iterationWeights: [1.0, 1.0, 1.0],
nodeSelfInfluence: 0.1,

// Node filter (Same as write)
nodeLabels: ['Episode', 'Chunk', 'Topic', 'Concept', 'Technology', 'ReferenceLink']
}
)
YIELD
nodeCount,
nodePropertiesWritten,
mutateMillis,
configuration
RETURN
nodeCount,
nodePropertiesWritten,
mutateMillis,
configuration;

#### Node Similarity, calculate and mutate - in-memory graph projection with SIMILAR (relation) and score (node property)

In [ ]:
CALL gds.nodeSimilarity.write('EpisodeSimilarityGraph', {
    writeRelationshipType: 'SIMILAR', 
    writeProperty: 'score' 
})
YIELD nodesCompared, relationshipsWritten

#### Experimental cyphers

#### List All relationships in DB

In [ ]:
MATCH ()-[r]->()
RETURN DISTINCT type(r) AS RelationshipType
ORDER BY RelationshipType

### List All Nodes and their corresponding relationship list from schema

In [ ]:
CALL apoc.meta.schema()
YIELD value
UNWIND keys(value) AS label

// Carry forward 'label' and 'value' before filtering
WITH label, value
WHERE value[label].type = 'node'

// Extract relationship information for each node label
RETURN label AS NodeLabel,
       [
           r IN keys(value[label].relationships) | 
           value[label].relationships[r].direction + ' ' + r 
       ] AS Relationships
ORDER BY NodeLabel

In [ ]:
MATCH (n)-[r]-(m)
// Match all relationships

WITH DISTINCT labels(n) AS NodeLabels, type(r) AS RelationshipType, labels(m) AS ConnectedLabels
UNWIND NodeLabels AS NodeLabel

RETURN NodeLabel, 
       collect(DISTINCT RelationshipType) AS ConnectedRelationships
ORDER BY NodeLabel

#### List All Nodes and their Properties in DB

In [ ]:
CALL apoc.meta.schema()
YIELD value

UNWIND keys(value) AS label // Iterate over all node labels

// *** FIX: Use WITH to carry 'label' and 'value' forward and THEN filter ***
WITH label, value
WHERE value[label].type = 'node' // Filter for only node labels (not relationships)

RETURN label AS NodeLabel,
       [key IN keys(value[label].properties) | key] AS PropertyKeys
ORDER BY NodeLabel

#### Project Graph with Chunk

In [ ]:
// 1. Define the Node Projection
//    - Episode needs 'title' (String) for display and labeling.
//    - Chunk needs 'embedding' (List of Float) for calculation.
//
// ⚠️ Note: We explicitly define the property type here to avoid type inference errors.
WITH {
    Episode: { 
        label: 'Episode', 
        properties: {
            title: { type: 'STRING' } 
        }
    }, 
    Chunk: { 
        label: 'Chunk', 
        properties: {
            embedding: { type: 'LIST OF FLOAT' } 
        }
    } 
} AS node_projection

// 2. Define the Relationship Projection
//    - HAS_CHUNK is the relationship type.
//    - We assume the relationship is directed (Episode -> Chunk), but we can change 
//      'orientation' if needed. For embeddings, 'UNDIRECTED' is often better 
//      to allow influence to flow both ways.
WITH node_projection, {
    HAS_CHUNK: { 
        type: 'HAS_CHUNK', 
        orientation: 'UNDIRECTED' // Use UNDIRECTED for better context flow
    } 
} AS relationship_projection

// 3. Estimate memory for the projection (optional, but good practice)
CALL gds.graph.project.estimate(
    node_projection,
    relationship_projection
)
YIELD requiredMemory
RETURN requiredMemory;

#### Estimate memory for projection

In [ ]:
// 1. Define the Node Projection
//    - Episode needs 'title' (String) for display and labeling.
//    - Chunk needs 'embedding' (List of Float) for calculation.
//
// ⚠️ Note: We explicitly define the property type here to avoid type inference errors.
WITH {
    Episode: { 
        label: 'Episode', 
        properties: {
            title: { type: 'STRING' } 
        }
    }, 
    Chunk: { 
        label: 'Chunk', 
        properties: {
            embedding: { type: 'LIST OF FLOAT' } 
        }
    } 
} AS node_projection

// 2. Define the Relationship Projection
//    - HAS_CHUNK is the relationship type.
//    - We assume the relationship is directed (Episode -> Chunk), but we can change 
//      'orientation' if needed. For embeddings, 'UNDIRECTED' is often better 
//      to allow influence to flow both ways.
WITH node_projection, {
    HAS_CHUNK: { 
        type: 'HAS_CHUNK', 
        orientation: 'UNDIRECTED' // Use UNDIRECTED for better context flow
    } 
} AS relationship_projection

// 3. Estimate memory for the projection (optional, but good practice)
CALL gds.graph.project.estimate(
    node_projection,
    relationship_projection
)
YIELD requiredMemory
RETURN requiredMemory;

#### Experiment with FastRP Mutate call - Fast RP Mutate call to cerate episodeEmbedding

In [ ]:
CALL gds.fastRP.mutate(
    "episode-content-embeddings", 
    {
        //Limit validation of featureProperties to 'Chunk' nodes
        nodeLabels: ['Chunk'], 
        // Name of the new node property to be stored in the GDS graph
        mutateProperty: 'episodeEmbedding', 
        
        // Define the size of the output vector
        embeddingDimension: 128, 
        
        // Use the existing Chunk 'embedding' as the input feature
        featureProperties: ['embedding'], 
        
        // Set to 1.0 to base the initial projection entirely on the feature properties
        propertyRatio: 1.0, 
        
        // Set the random seed for consistency between runs
        randomSeed: 42,
        
        // Iteration weights control feature propagation; the default should suffice
        iterationWeights: [0.0, 1.0, 1.0] 
    }
)
YIELD 
    nodeCount, 
    nodePropertiesWritten, 
    mutateMillis,
    configuration
RETURN 
    nodeCount, 
    nodePropertiesWritten, 
    mutateMillis,
    configuration;

In [ ]:
MATCH (p:Podcast)-[:HAS_EPISODE]->(e:Episode {number:473})-[:HAS_TOPIC]->(t:Topic)
OPTIONAL MATCH (t)-[:COVERS_TECHNOLOGY]->(existingTech:Technology)
WITH e, t, collect(existingTech.name) AS existingTechnologies
UNWIND ["Lakehouse ecosystem", "Snowflake", "Database"] as techName
MERGE (newTech:Technology {name: techName})
MERGE (t)-[:COVERS_TECHNOLOGY]->(newTech)
RETURN 
  e.name AS episode,
  t.name AS topic,
  existingTechnologies,
  newTech.name AS addedTechnology;


#### Remove a Technology relation ONLy if it exist


In [ ]:
OPTIONAL MATCH (p:Podcast {title: "Data Engineering Podcast"})
      -[:HAS_EPISODE]->(e:Episode {number: 480})
      -[:HAS_TOPIC]->(t:Topic)
      -[rel:COVERS_TECHNOLOGY]->(tech:Technology {name: "Snowflake"})
WITH rel, p, e, t, tech
WHERE rel IS NOT NULL
DELETE rel
RETURN p.title AS podcast, e.name AS episode, t.name AS topic, tech.name AS removedTechnology;


#### Remove duplicate connection to episode

In [ ]:

//Remove connection to episode: number: 2025040307, number: 1654
MATCH (p:Podcast {title:"Data Engineering Podcast"})-[r:HAS_EPISODE]->(e:Episode)
RETURN p,r,e

// Delete the HAS_EPISODE relationship for the specific episode
//MATCH (p:Podcast {title: "Data Engineering Podcast"})-[r:HAS_EPISODE]->(e:Episode {number: 1654})
//DELETE r;

#### //Nodes query:

//MATCH (n) WHERE n:Episode OR n:Technology: grabs all Episode and Technology nodes.

//RETURN id(n) AS id, labels(n) AS labels: assigns the node id and includes the node labels as a property for reference.

In [ ]:
 MATCH (n)
  WHERE n:Episode OR n:Technology
  RETURN id(n) AS id, n.name AS node_name,
         labels(n) AS node_type

#### //Relationships query:

//Projects Episode → Technology edges via Topic relationships.

//id(e) AS source, id(t) AS target: required by GDS for projection.

//"COVERS_TECHNOLOGY" AS type: assigns a type to the virtual relationship.

In [ ]:
MATCH (e:Episode)-[:HAS_TOPIC]->(:Topic)-[:COVERS_TECHNOLOGY]->(t:Technology)
RETURN id(e) AS source, e.name AS source_name, id(t) AS target, t.name AS target_name, "COVERS_TECHNOLOGY" AS type


#### //Cypher Projection (Aggregation Method)
//This query first aggregates the relationship path using standard     
//Cypher and then projects the resulting data using the gds.graph.project function      
//within a WITH clause.

In [ ]:
MATCH (e1:Episode)-[:HAS_TOPIC]->(:Topic)-[:COVERS_TECHNOLOGY]->(t:Technology)
// Match back to another Episode that covers the same Technology
MATCH (e2:Episode)-[:HAS_TOPIC]->(:Topic)-[:COVERS_TECHNOLOGY]->(t)
// Ensure we don't count self-loops and only count each pair once
WHERE id(e1) < id(e2)

// Aggregate the common technology count between the two episodes
WITH e1, e2, count(t) AS commonTechCount

// Project the graph using the aggregation result in the WITH clause
WITH gds.graph.project(
    'episodeTechGraph',
    e1, // The source node (Episode)
    e2, // The target node (Episode)
    { 
        relationshipType: 'SHARES_TECHNOLOGY', // Define a new relationship type
        relationshipProperties: {
            // Project the count as a relationship property
            commonTechCount: commonTechCount
        }
    }
) AS g
RETURN
  g.graphName AS graph, 
  g.nodeCount AS nodeCount, 
  g.relationshipCount AS relationshipCount;

#### Find Similar Episodes (Node Similarity)
//Once the graph is projected, the easiest way to find similarity is to use the 
//commonTechCount as a weight in the Node Similarity algorithm, although in this specific 
//projection, since the e1 and e2 nodes share a direct relationship, standard 
//Node Similarity isn't necessary. The similarity is already encoded in the commonTechCount.
//Instead, you can just query the resulting graph using the aggregation data you created:

In [ ]:
// Query the projected graph for the most similar episodes
CALL gds.graph.list('episodeTechGraph')
YIELD graphName
// We don't need the similarity algorithm; we just need the aggregated count
// However, the gds.graph.project in the WITH clause only creates the graph; it doesn't return the raw data.

// To see the similar episodes, you must run the initial MATCH/WITH/RETURN WITHOUT the GDS project call:
MATCH (e1:Episode)-[:HAS_TOPIC]->(:Topic)-[:COVERS_TECHNOLOGY]->(t:Technology)
MATCH (e2:Episode)-[:HAS_TOPIC]->(:Topic)-[:COVERS_TECHNOLOGY]->(t)
WHERE id(e1) < id(e2)
WITH e1, e2, count(t) AS commonTechCount
RETURN e1.name AS Episode1, e2.name AS Episode2, commonTechCount
ORDER BY commonTechCount DESC
LIMIT 20;

#### //To combine these steps and automate the process for the top similar pairs, you will use a single Cypher query that performs the aggregation and then, for the top pairs, uses CALL {} IN TRANSACTIONS (or simply another WITH clause) to find the details of the shared technologies.

The top 3 similar pairs, the final query will show all three pairs and their corresponding lists of shared technologies. Note: Neo4j does not allow creating a "graph" directly in the return, but it can return the nodes and relationships needed for visualization.

Here is the single, combined Cypher query:

In [ ]:
// 1. Find the top 3 most similar episode pairs based on shared Technology count
MATCH (e1:Episode)-[:HAS_TOPIC]->(:Topic)-[:COVERS_TECHNOLOGY]->(t1:Technology)
MATCH (e2:Episode)-[:HAS_TOPIC]->(:Topic)-[:COVERS_TECHNOLOGY]->(t2:Technology)
WHERE id(e1) < id(e2) AND t1 = t2
WITH e1, e2, count(t1) AS commonTechCount
ORDER BY commonTechCount DESC
LIMIT 3

// 2. For each of the top 3 pairs, find the actual Technology names
MATCH (e1)-[:HAS_TOPIC]->(:Topic)-[:COVERS_TECHNOLOGY]->(t:Technology)
MATCH (e2)-[:HAS_TOPIC]->(:Topic)-[:COVERS_TECHNOLOGY]->(t)
WITH e1, e2, commonTechCount, collect(DISTINCT t.name) AS sharedTechnologies, collect(t) AS techNodes

// 3. Collect the episode pair details
WITH collect({
    episode1: e1.name,
    episode2: e2.name,
    count: commonTechCount,
    technologies: sharedTechnologies,
    techNodes: techNodes,
    e1Node: e1,
    e2Node: e2
}) AS results

// 4. Return the list for the table and the elements for the graph visualization
UNWIND results AS result
RETURN
    // 1) List of similar episodes and common tech count
    result.episode1 AS Episode1,
    result.episode2 AS Episode2,
    result.count AS CommonTechCount,
    result.technologies AS SharedTechnologies,
    
    // 2) Nodes and relationships for graph visualization
    result.e1Node,
    result.e2Node,
    result.techNodes

#### TODO: TRY //Creating IS_SIMILAR_TO Cypher Query to Create the IS_SIMILAR_TO Relationship
This single query performs the following actions:

Finds episode pairs that share at least one technology.

Counts the shared technologies (commonTechCount).

Filters to only the top similar pairs (e.g., top 10) to avoid creating relationships between every loosely related episode.

Creates (or Merges) a new directed relationship, IS_SIMILAR_TO, between the episodes.

Sets the commonTechCount as a property (shared_tech_count) on the new relationship.

In [ ]:
// 1. Find pairs of episodes (e1, e2) that share the same technology (t)
MATCH (e1:Episode)-[:HAS_TOPIC]->(:Topic)-[:COVERS_TECHNOLOGY]->(t:Technology)
MATCH (e2:Episode)-[:HAS_TOPIC]->(:Topic)-[:COVERS_TECHNOLOGY]->(t)
// Ensure e1 and e2 are different nodes and process each pair only once
WHERE id(e1) < id(e2)

// 2. Aggregate the count of shared technologies
WITH e1, e2, count(t) AS commonTechCount
ORDER BY commonTechCount DESC

// 3. Filter for a relevance threshold (e.g., at least 2 shared technologies)
//    and limit the total number of new relationships to create (e.g., top 100)
WHERE commonTechCount >= 2
LIMIT 100

// 4. Create the new weighted relationship
MERGE (e1)-[r:IS_SIMILAR_TO]->(e2)
SET r.shared_tech_count = commonTechCount
SET r.date_created = datetime()

// 5. Return the result of the new relationship
RETURN e1.name AS Episode1, e2.name AS Episode2, r.shared_tech_count AS SimilarityScore;

## Exploratory Cypher

In [ ]:
// Cypher projection with node labels
MATCH (e:Episode)-[:HAS_TOPIC]->(:Topic)-[:COVERS_TECHNOLOGY]->(t:Technology)
WITH e AS source, t AS target
WITH gds.graph.project(
  'episodeTechGraph',
  source, target,
  { nodeLabels: { Episode: 'Episode', Technology: 'Technology' } }
) AS g
RETURN g.graphName, g.nodeCount, g.relationshipCount;


In [ ]:
from cProfile import label


//Neo4j OLD - Cypher projection
// Project a virtual Episode–Technology graph via Topic relationships
CALL gds.graph.project.cypher(
  'episode-tech-projection',
  
  // Nodes: include both Episodes and Technologies
  'MATCH (n) WHERE n:Episode OR n:Technology RETURN id(n) AS id, label(n) as label,
  
  // Relationships: Episode–Technology via Topic
  '
  MATCH (e:Episode)-[:HAS_TOPIC]->(:Topic)-[:COVERS_TECHNOLOGY]->(t:Technology)
  RETURN id(e) AS source, id(t) AS target, "COVERS_TECHNOLOGY" AS type
  '
)
YIELD graphName, nodeCount, relationshipCount
RETURN graphName, nodeCount, relationshipCount;


In [ ]:
//Neo4j New Neo4j 5.23 + GDS 3.0 and later deprecated CALL gds.graph.project.cypher.
// ✅ Modern syntax using gds.graph.project() aggregation form
RETURN gds.graph.project(
  'episode-tech-projection',  // graph name

  // Node projection: Episodes and Technologies
  {
    Episode: {},
    Technology: {}
  },

  // Relationship projection: Episode–Technology via Topic
  {
    COVERS_TECHNOLOGY: {
      type: 'COVERS_TECHNOLOGY',
      orientation: 'UNDIRECTED',
      properties: {}
    }
  }
) AS graphInfo;


In [ ]:
//since your relationships between Episode and Technology are indirect (through Topic), you need to materialize or virtually expose them first.
//You can do that using graph.project.cypher inlined via gds.graph.project, like so:

// ✅ Cypher-based virtual projection (modern function style)
RETURN gds.graph.project(
  'episode-tech-projection',
  {
    nodeQuery: 'MATCH (n) WHERE n:Episode OR n:Technology RETURN id(n) AS id',
    relationshipQuery: '
      MATCH (e:Episode)-[:HAS_TOPIC]->(:Topic)-[:COVERS_TECHNOLOGY]->(t:Technology)
      RETURN id(e) AS source, id(t) AS target, "COVERS_TECHNOLOGY" AS type
    '
  }
) AS graphInfo;
